# Stage B / NB 05 — Frozen encoder agent (A5, CXformer-base)

Protocol reference: Section 6.1 agent A5; Section 6.2 experiments **E0d, E0e, E0f**;
research question **RQ3**. Answers part of referee 2d ("compare with established CXR models").

## Why this notebook runs first in Stage B

It is by far the cheapest arm (~1 GPU-hour) and it sets the bar everything else must clear.
`m42-health/CXformer-base` is 87M parameters, frozen, with only a small trained head. If a
linear probe on frozen features matches the 4B LoRA-adapted VLMs, that is a finding the paper
must report rather than bury — and it is much better to learn it in week 2 than in week 8.
Protocol gate **G2** exists for exactly this decision.

Since `google/cxr-foundation` was withdrawn (D2a), A5 is the **sole** frozen-encoder agent and
carries RQ3 alone. It also occupies the role BiomedCLIP played in the rejected version, so its
leave-one-out arm (E1-L5) is what answers referee 1.2's question about whether a dedicated
embedding agent contributes anything.

## The three arms

| arm | head | views | what it isolates |
| --- | --- | --- | --- |
| E0d | linear probe | V0 | how much is linearly decodable from frozen features |
| E0e | 2-layer MLP | V0 | whether non-linearity on the same features helps |
| E0f | per-lung ordinal (CORAL) | V2L + V2R | whether anatomy-aware decomposition helps a **non-generative** model too |

E0f is the interesting one. If the anatomy-aware decomposition of E4 only helps generative
VLMs, it is plausibly compensating for a hard counting task rather than supplying a genuine
inductive bias. If it helps a frozen linear head as well, the inductive-bias claim in
referee 2a's rebuttal is much stronger.

## Outputs (under `stage_B/nb05_frozen_encoder/`)
- `embeddings/{view}.npz` — cached frozen features, keyed by `image_key`
- `predictions_frozen.jsonl` — shared prediction schema (see `cxr_metrics.PREDICTION_FIELDS`)
- `per_fold_metrics.json`, `cross_validation_aggregate_95ci.csv`, `arm_summary.csv`
- `external_predictions.jsonl`, `run_config.json`, `gate_nb05.json`

## Gate
- Embeddings exist for every internal image and every requested view.
- Out-of-fold coverage is exactly 100% of the internal cohort — each image predicted once, by
  the fold that held it out.
- AUROC is computable (both PCR classes present) in every fold.

## 1. Imports, seeds, and the Stage A path contract

In [ ]:
import gc
import json
import math
import os
import random
import sys
import time
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Shared metric definitions. Table 2 is only a valid comparison if every arm uses these.
_METRICS_SEARCH = [Path.cwd(), Path.cwd().parent, Path.cwd() / "stage_B",
                   Path.cwd().parent / "stage_B"]
for _candidate in _METRICS_SEARCH:
    if (_candidate / "cxr_metrics.py").is_file():
        sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(
        "cxr_metrics.py not found. It must sit beside the Stage B notebooks; every arm in "
        f"Table 2 depends on its metric definitions. Searched: {_METRICS_SEARCH}")
import cxr_metrics as cm

SEED = 42
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# ---- Stage A path contract -------------------------------------------------------------
FALLBACK_STAGE_A_DIR = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
PATHS_JSON_CANDIDATES = [
    FALLBACK_STAGE_A_DIR / "nb00_environment" / "stage_a_paths.json",
    Path.cwd() / "stage_a_paths.json",
    Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json",
]
stage_paths = None
for candidate in PATHS_JSON_CANDIDATES:
    if candidate.is_file():
        stage_paths = json.loads(candidate.read_text(encoding="utf-8"))
        print("Path contract:", candidate)
        break
if stage_paths is None:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
INTERNAL_VALIDATION_FRACTION = 0.10   # matches the tested LoRA notebooks

print("Stage B output root:", STAGE_B_DIR)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| BF16:", torch.cuda.is_bf16_supported())

## 2. Configuration

`EMBEDDING_VIEWS` drives which cached NB 04 views get encoded. V0 and the two regional views
are required by the three arms; V1 is optional and cheap to add if you later want a
thorax-crop frozen arm to sit alongside E4b.

In [ ]:
import hashlib

NB05_DIR = STAGE_B_DIR / "nb05_frozen_encoder"
EMBEDDING_DIR = NB05_DIR / "embeddings"
for directory in [NB05_DIR, EMBEDDING_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

ENCODER_ID = "m42-health/CXformer-base"
ENCODER_REVISION = MODEL_REVISIONS.get(ENCODER_ID)

# NB 04 view column -> short name used in file names and the "view" prediction field.
EMBEDDING_VIEWS = OrderedDict([
    ("v0", "v0_image"),
    ("v2_left", "v2_left_image"),
    ("v2_right", "v2_right_image"),
])
INCLUDE_V1_THORAX = False
if INCLUDE_V1_THORAX:
    EMBEDDING_VIEWS["v1_thorax"] = "v1_thorax_image"

EMBEDDING_BATCH_SIZE = 32
# The embedding cache is flushed every this-many images, so an interruption costs at most
# this much work instead of the whole view. Writes are atomic via a .tmp rename.
EMBEDDING_FLUSH_EVERY = 512
EMBEDDING_POOLING = "cls_plus_mean"   # "cls", "mean", or "cls_plus_mean" (concatenated)

# Head training. Deliberately small: these are probes, not models. Their job is to measure how
# much the FROZEN representation already contains, so a heavy head would confound the answer.
HEAD_EPOCHS = 60
HEAD_BATCH_SIZE = 256
HEAD_LEARNING_RATE = 1e-3
HEAD_WEIGHT_DECAY = 1e-4
HEAD_HIDDEN_DIM = 512
HEAD_DROPOUT = 0.2
HEAD_PATIENCE = 10
MC_DROPOUT_SAMPLES = 20               # uncertainty for the MLP/ordinal arms

# E8 hooks (protocol Section 6.2). Off by default so E0d/E0e/E0f are clean baselines; NB 08
# owns the imbalance sweep.
USE_CLASS_BALANCED_LOSS = False
USE_LOGIT_ADJUSTMENT = False

RUN_ARMS = ["E0d_linear_v0", "E0e_mlp_v0", "E0f_ordinal_lungs"]
RUN_EXTERNAL = True
SMOKE_TEST_IMAGES = None              # set e.g. 64 for a fast wiring check, then None

print("Encoder:", ENCODER_ID, "revision:", ENCODER_REVISION)
print("Views to embed:", list(EMBEDDING_VIEWS))
print("Arms:", RUN_ARMS)
print("Output:", NB05_DIR)

## 3. Load the cohort table (labels + folds + cached view paths)

In [ ]:
def load_view_index():
    path = NB04_DIR / "view_index.csv"
    if not path.is_file():
        raise FileNotFoundError(f"{path} not found. Run Stage A NB 04 first.")
    frame = pd.read_csv(path)
    print(f"view_index.csv: {len(frame):,} rows, cohorts={dict(Counter(frame['cohort']))}")
    return frame


def load_folds():
    path = FOLD_DEF_DIR / "midrc_folds_v2.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} not found. Run Stage A NB 02 first. Do NOT fall back to the legacy "
            "multi_task_CV folds: they leak at study level."
        )
    frame = pd.read_csv(path)
    print(f"midrc_folds_v2.csv: {len(frame):,} images, "
          f"{frame['group_id'].nunique():,} groups, folds={dict(sorted(Counter(frame['fold']).items()))}")
    return frame


def build_cohort_table():
    # One row per image: labels + fold + every cached view path. This is the single table
    # every Stage B notebook trains and predicts from.
    views = load_view_index()
    folds = load_folds()

    internal = folds.merge(
        views[views["cohort"] == "MIDRC"].drop(columns=["held_out_fold"], errors="ignore"),
        on="filename", how="inner", suffixes=("", "_view"),
    )
    if len(internal) != len(folds):
        missing = set(folds["filename"]) - set(internal["filename"])
        raise RuntimeError(
            f"{len(missing)} fold images have no NB 04 localization row (e.g. "
            f"{sorted(missing)[:5]}). Re-run NB 04 with MAX_IMAGES=None."
        )
    internal["mrale_right"] = (internal["extent_right_numerical"]
                               * internal["density_right_numerical"])
    internal["mrale_left"] = (internal["extent_left_numerical"]
                              * internal["density_left_numerical"])
    internal["is_external"] = False

    external_rows = []
    external_dir = NB03_DIR / "external_manifests"
    if external_dir.is_dir():
        for manifest_path in sorted(external_dir.glob("*_manifest.csv")):
            frame = pd.read_csv(manifest_path)
            if "status" in frame.columns:
                frame = frame[frame["status"] == "OK"]
            if not len(frame):
                continue
            cohort = str(frame["cohort"].iloc[0])
            merged = frame.merge(
                views[views["cohort"] == cohort][
                    ["filename", "v0_image", "v1_thorax_image", "v2_left_image",
                     "v2_right_image", "left_box", "right_box", "any_fallback"]
                ],
                on="filename", how="inner",
            )
            merged["fold"] = -1
            merged["is_external"] = True
            merged["group_id"] = "external::" + merged["filename"].astype(str)
            for column in ["mrale_total_annotated", "mrale_right", "mrale_left",
                           "extent_right_numerical", "density_right_numerical",
                           "extent_left_numerical", "density_left_numerical"]:
                if column not in merged.columns:
                    merged[column] = np.nan
            if "mrale_total" in merged.columns:
                merged["mrale_total_annotated"] = merged["mrale_total"]
            external_rows.append(merged)

    table = pd.concat([internal] + external_rows, ignore_index=True, sort=False)
    table["image_key"] = table.apply(
        lambda row: f"{row.get('cohort', 'MIDRC')}::{row['filename']}", axis=1)
    print()
    print(f"Cohort table: {len(table):,} rows "
          f"({int((~table['is_external']).sum()):,} internal, "
          f"{int(table['is_external'].sum()):,} external)")
    return table


def grouped_inner_split(subset, fraction, seed):
    # Group-aware inner validation split, same construction as the tested notebooks: whole
    # groups move together so the inner split cannot leak either.
    groups = sorted(subset["group_id"].astype(str).unique())
    rng = random.Random(seed)
    rng.shuffle(groups)
    n_validation = max(1, round(len(groups) * fraction))
    validation_groups = set(groups[:n_validation])
    is_validation = subset["group_id"].astype(str).isin(validation_groups)
    train, validation = subset[~is_validation], subset[is_validation]
    assert not (set(train["group_id"]) & set(validation["group_id"]))
    return train, validation


def ground_truth_fields(row):
    def maybe_int(value):
        return None if value is None or (isinstance(value, float) and math.isnan(value)) else int(value)
    covid = row.get("covid_positive")
    if isinstance(covid, float) and math.isnan(covid):
        covid = None
    return {
        "gt_covid": covid if covid in {"Yes", "No"} else None,
        "gt_mrale_total": maybe_int(row.get("mrale_total_annotated")),
        "gt_mrale_right": maybe_int(row.get("mrale_right")),
        "gt_mrale_left": maybe_int(row.get("mrale_left")),
        "gt_extent_right": maybe_int(row.get("extent_right_numerical")),
        "gt_density_right": maybe_int(row.get("density_right_numerical")),
        "gt_extent_left": maybe_int(row.get("extent_left_numerical")),
        "gt_density_left": maybe_int(row.get("density_left_numerical")),
    }


def evaluate_arm(rows, label):
    # Single entry point for metrics, so every arm in Table 2 is scored identically.
    covid_rows = [row for row in rows if row.get("gt_covid") is not None]
    metrics = {"arm": label, "n_rows": len(rows)}
    if covid_rows:
        metrics["covid"] = cm.classification_metrics(
            [row["gt_covid"] for row in covid_rows],
            [row.get("covid_pred") for row in covid_rows],
            [row.get("covid_score") for row in covid_rows],
        )
    mrale_rows = [row for row in rows if row.get("gt_mrale_total") is not None]
    if mrale_rows:
        metrics["mrale"] = cm.mrale_metrics(mrale_rows)
    metrics["output"] = cm.localization_free_metrics(rows)
    return metrics


def print_arm_summary(metrics):
    covid = metrics.get("covid", {})
    mrale = metrics.get("mrale", {})
    print(f"  {metrics['arm']:<34} "
          f"AUROC={covid.get('auroc', float('nan')):.4f} "
          f"balAcc={covid.get('balanced_accuracy', float('nan')):.4f} "
          f"spec={covid.get('specificity', float('nan')):.4f} | "
          f"mRALE MAE={mrale.get('mae', float('nan')):.3f} "
          f"QWK={mrale.get('qwk', float('nan')):.4f} "
          f"cov={mrale.get('coverage', float('nan')):.3f}")

In [ ]:
cohort = build_cohort_table()
if SMOKE_TEST_IMAGES is not None:
    internal_sample = cohort[~cohort["is_external"]].groupby("fold", group_keys=False).head(
        max(2, SMOKE_TEST_IMAGES // N_FOLDS))
    external_sample = cohort[cohort["is_external"]].head(8)
    cohort = pd.concat([internal_sample, external_sample], ignore_index=True)
    print(f"SMOKE TEST: reduced to {len(cohort)} rows")

internal_cohort = cohort[~cohort["is_external"]].reset_index(drop=True)
external_cohort = cohort[cohort["is_external"]].reset_index(drop=True)
print()
print("PCR balance (internal):", dict(internal_cohort["covid_positive"].value_counts()))
print("Severity bands (internal):", dict(internal_cohort["severity_band"].value_counts()))

## 4. Extract frozen embeddings

One forward pass per (image, view), cached to `.npz` and keyed by `image_key`. Re-running skips
what already exists, so this is safe to interrupt.

The encoder is loaded with `trust_remote_code=True`: CXformer ships a DINOv2-derived
implementation. If that fails on this node, the fallback path tries `timm` — and if both fail,
the notebook stops rather than silently substituting a different backbone, because "CXformer"
in Table 2 must actually mean CXformer.

In [ ]:
from PIL import Image

Image.MAX_IMAGE_PIXELS = None


def load_encoder():
    errors = {}
    kwargs = {"trust_remote_code": True}
    if ENCODER_REVISION:
        kwargs["revision"] = ENCODER_REVISION
    try:
        from transformers import AutoImageProcessor, AutoModel
        processor = AutoImageProcessor.from_pretrained(ENCODER_ID, **kwargs)
        model = AutoModel.from_pretrained(ENCODER_ID, **kwargs)
        model.eval()
        return model, processor, "transformers.AutoModel"
    except Exception as exc:
        errors["transformers"] = f"{type(exc).__name__}: {exc}"
    try:
        import timm
        model = timm.create_model(f"hf_hub:{ENCODER_ID}", pretrained=True, num_classes=0)
        model.eval()
        config = timm.data.resolve_data_config({}, model=model)
        transform = timm.data.create_transform(**config, is_training=False)
        return model, transform, "timm"
    except Exception as exc:
        errors["timm"] = f"{type(exc).__name__}: {exc}"
    raise RuntimeError(
        "CXformer-base could not be loaded by either backend. Table 2's frozen-encoder row "
        f"cannot be produced without it, and substituting another backbone would misreport "
        f"the comparison. Errors: {json.dumps(errors, indent=2)}"
    )


@torch.inference_mode()
def encode_batch(model, processor, backend, image_paths, device):
    images = []
    for path in image_paths:
        with Image.open(path) as handle:
            images.append(handle.convert("RGB"))
    if backend == "timm":
        batch = torch.stack([processor(image) for image in images]).to(device)
        features = model.forward_features(batch)
        if features.ndim == 3:
            cls_token, patch_tokens = features[:, 0], features[:, 1:].mean(dim=1)
        else:
            cls_token = patch_tokens = features.flatten(1)
    else:
        inputs = processor(images=images, return_tensors="pt")
        inputs = {key: value.to(device) for key, value in inputs.items()}
        outputs = model(**inputs)
        hidden = getattr(outputs, "last_hidden_state", None)
        if hidden is None:
            hidden = outputs[0]
        if hidden.ndim == 3:
            cls_token, patch_tokens = hidden[:, 0], hidden[:, 1:].mean(dim=1)
        else:
            cls_token = patch_tokens = hidden.flatten(1)
    for image in images:
        image.close()

    if EMBEDDING_POOLING == "cls":
        pooled = cls_token
    elif EMBEDDING_POOLING == "mean":
        pooled = patch_tokens
    else:
        pooled = torch.cat([cls_token, patch_tokens], dim=-1)
    return pooled.float().cpu().numpy()


# The cache key must cover everything that changes an embedding, not just the view name.
# Without this, switching EMBEDDING_POOLING or the encoder revision silently reuses vectors
# computed under the old setting -- the same trap that NB 08's entity cache had.
EMBEDDING_FINGERPRINT = hashlib.sha256(json.dumps({
    "encoder": ENCODER_ID, "revision": ENCODER_REVISION,
    "pooling": EMBEDDING_POOLING, "views": list(EMBEDDING_VIEWS),
}, sort_keys=True).encode()).hexdigest()[:16]
print("Embedding-cache fingerprint:", EMBEDDING_FINGERPRINT)


def embed_view(view_name, path_column, model, processor, backend, device):
    cache_path = EMBEDDING_DIR / f"{view_name}.npz"
    cached = {}
    if cache_path.is_file():
        with np.load(cache_path, allow_pickle=False) as archive:
            keys = archive["image_keys"].tolist()
            matrix = archive["features"]
            stored = (str(archive["fingerprint"].item())
                      if "fingerprint" in archive else None)
        if stored is None:
            # A cache written before fingerprinting existed. Accept it rather than discarding
            # a completed run, but say so: it cannot be verified against the current settings.
            cached = {str(key): matrix[index] for index, key in enumerate(keys)}
            print(f"  [{view_name}] legacy cache accepted ({len(cached):,} vectors, no "
                  "fingerprint). Delete the .npz if the encoder or pooling has changed since "
                  "it was written.")
        elif stored != EMBEDDING_FINGERPRINT:
            print(f"  [{view_name}] fingerprint mismatch (cached {stored}, now "
                  f"{EMBEDDING_FINGERPRINT}): encoder, revision or pooling changed. "
                  "Recomputing rather than reusing stale vectors.")
        else:
            cached = {str(key): matrix[index] for index, key in enumerate(keys)}
            print(f"  [{view_name}] cache hit: {len(cached):,}")

    todo = []
    for _, row in cohort.iterrows():
        path = row.get(path_column)
        if not isinstance(path, str) or not path or path == "nan":
            continue
        if str(row["image_key"]) in cached:
            continue
        todo.append((str(row["image_key"]), path))

    def flush(label):
        # Atomic write: a crash during np.savez would otherwise leave a truncated cache that
        # looks valid on the next run.
        keys = sorted(cached)
        temporary = cache_path.with_suffix(".npz.tmp")
        np.savez_compressed(
            temporary,
            image_keys=np.asarray(keys),
            fingerprint=np.asarray(EMBEDDING_FINGERPRINT),
            features=np.stack([cached[key] for key in keys]).astype(np.float32),
        )
        temporary.replace(cache_path)
        print(f"  [{view_name}] {label}: cache now holds {len(keys):,} vectors")

    if todo:
        print(f"  [{view_name}] encoding {len(todo):,} images")
        started = time.perf_counter()
        since_flush = 0
        for offset in range(0, len(todo), EMBEDDING_BATCH_SIZE):
            chunk = todo[offset:offset + EMBEDDING_BATCH_SIZE]
            try:
                features = encode_batch(
                    model, processor, backend, [item[1] for item in chunk], device)
            except Exception as exc:
                # Persist what has been computed before surfacing the failure, so a bad image
                # partway through does not discard the preceding work.
                if since_flush:
                    flush("partial flush after error")
                raise RuntimeError(
                    f"[{view_name}] batch at offset {offset} failed: {exc}") from exc
            for (key, _), vector in zip(chunk, features):
                cached[key] = vector
            since_flush += len(chunk)
            if since_flush >= EMBEDDING_FLUSH_EVERY:
                flush("checkpoint")
                since_flush = 0
            if (offset // EMBEDDING_BATCH_SIZE) % 10 == 0:
                done = min(offset + EMBEDDING_BATCH_SIZE, len(todo))
                rate = done / max(time.perf_counter() - started, 1e-6)
                print(f"    {done}/{len(todo)} ({rate:.1f} img/s)")
        flush("final")
    return cached


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder, encoder_processor, encoder_backend = load_encoder()
encoder = encoder.to(device)
n_encoder_parameters = sum(p.numel() for p in encoder.parameters())
print(f"Loaded CXformer via {encoder_backend}: "
      f"{n_encoder_parameters / 1e6:.1f}M parameters (frozen)")
print()

embeddings = {}
for view_name, path_column in EMBEDDING_VIEWS.items():
    embeddings[view_name] = embed_view(
        view_name, path_column, encoder, encoder_processor, encoder_backend, device)

del encoder
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

embedding_dim = len(next(iter(embeddings["v0"].values())))
print()
print(f"Embedding dimension ({EMBEDDING_POOLING}): {embedding_dim}")
for view_name, store in embeddings.items():
    print(f"  {view_name}: {len(store):,} vectors")

## 5. Heads

Three head types, all small on purpose.

- **Linear / MLP** — one COVID logit plus four ordinal component blocks (extent 0-4, density
  0-3, per lung). The mRALE total is reconstructed as
  `extent_right*density_right + extent_left*density_left`, so the arithmetic consistency
  constraint holds by construction rather than by hope.
- **Ordinal blocks use CORAL** (cumulative ordinal logits): K-1 binary outputs sharing one
  weight vector with ordered bias terms. This respects that extent 3 is closer to extent 4
  than to extent 0, which a plain 5-way softmax throws away — and mRALE extent and density
  are ordinal by definition.

The categorical distribution implied by the CORAL outputs gives both a hard integer prediction
and a soft expectation, which is what feeds `mrale_total_expected` and the uncertainty field.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class CoralBlock(nn.Module):
    # Cumulative-link ordinal head: P(y > k) for k = 0..K-2, monotone by shared weights.
    def __init__(self, in_features, n_classes):
        super().__init__()
        self.n_classes = n_classes
        self.projection = nn.Linear(in_features, 1, bias=False)
        self.thresholds = nn.Parameter(torch.zeros(n_classes - 1))

    def forward(self, features):
        return self.projection(features) + self.thresholds        # (batch, K-1) logits

    @staticmethod
    def loss(logits, targets, n_classes, weight=None):
        # targets: integer class index. Level k target is 1 when y > k.
        levels = torch.arange(n_classes - 1, device=logits.device)[None, :]
        binary = (targets[:, None] > levels).float()
        return F.binary_cross_entropy_with_logits(logits, binary, weight=weight)

    @staticmethod
    def to_distribution(logits):
        # P(y > k) -> categorical P(y = k), clamped to stay monotone under noise.
        greater = torch.sigmoid(logits)
        greater = torch.cummin(greater, dim=1).values
        ones = torch.ones_like(greater[:, :1])
        zeros = torch.zeros_like(greater[:, :1])
        upper = torch.cat([ones, greater], dim=1)
        lower = torch.cat([greater, zeros], dim=1)
        return (upper - lower).clamp_min(0.0)


class FrozenHead(nn.Module):
    """COVID logit + four ordinal component blocks on frozen features."""

    def __init__(self, in_features, hidden_dim=None, dropout=0.0):
        super().__init__()
        if hidden_dim:
            self.trunk = nn.Sequential(
                nn.Linear(in_features, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            )
            feature_dim = hidden_dim
        else:
            self.trunk = nn.Identity()
            feature_dim = in_features
        self.covid = nn.Linear(feature_dim, 1)
        self.extent_right = CoralBlock(feature_dim, 5)
        self.density_right = CoralBlock(feature_dim, 4)
        self.extent_left = CoralBlock(feature_dim, 5)
        self.density_left = CoralBlock(feature_dim, 4)

    def forward(self, features):
        hidden = self.trunk(features)
        return {
            "covid": self.covid(hidden).squeeze(-1),
            "extent_right": self.extent_right(hidden),
            "density_right": self.density_right(hidden),
            "extent_left": self.extent_left(hidden),
            "density_left": self.density_left(hidden),
        }


class LungOrdinalHead(nn.Module):
    """
    E0f: one shared per-lung head applied to V2L and V2R separately.

    Weight sharing is the point. The same head scores whichever lung it is shown, so the
    anatomy-aware arm cannot win merely by having twice the parameters -- it has to win because
    a single-lung view is genuinely easier to score than a whole-image view.
    """

    def __init__(self, in_features, hidden_dim=None, dropout=0.0):
        super().__init__()
        if hidden_dim:
            self.trunk = nn.Sequential(
                nn.Linear(in_features, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            )
            feature_dim = hidden_dim
        else:
            self.trunk = nn.Identity()
            feature_dim = in_features
        self.extent = CoralBlock(feature_dim, 5)
        self.density = CoralBlock(feature_dim, 4)
        # COVID is a whole-patient property, so it is read from the concatenated pair.
        self.covid = nn.Linear(feature_dim * 2, 1)

    def encode(self, features):
        return self.trunk(features)

    def lung_outputs(self, hidden):
        return {"extent": self.extent(hidden), "density": self.density(hidden)}

    def covid_logit(self, hidden_left, hidden_right):
        return self.covid(torch.cat([hidden_left, hidden_right], dim=-1)).squeeze(-1)


COMPONENTS = OrderedDict([
    ("extent_right", 5), ("density_right", 4), ("extent_left", 5), ("density_left", 4),
])
print("Heads defined. CORAL classes:", dict(COMPONENTS))

## 6. Fold training loop

Per fold: fit on the inner-train split, select the epoch on inner validation, predict the
held-out fold. Selection uses a single composite criterion — inner-validation mRALE MAE plus
`(1 - AUROC)` scaled to comparable magnitude — because these are joint COVID + mRALE heads and
picking on one task alone would quietly optimise the other away.

In [ ]:
def features_for(view_name, keys):
    store = embeddings[view_name]
    missing = [key for key in keys if key not in store]
    if missing:
        raise KeyError(f"[{view_name}] missing embeddings for {len(missing)} keys "
                       f"(e.g. {missing[:3]})")
    return torch.from_numpy(np.stack([store[key] for key in keys])).float()


def label_tensors(subset):
    covid = torch.tensor(
        [1.0 if value == "Yes" else 0.0 for value in subset["covid_positive"]], dtype=torch.float32)
    covid_mask = torch.tensor(
        [value in {"Yes", "No"} for value in subset["covid_positive"]], dtype=torch.bool)
    components, masks = {}, {}
    for component in COMPONENTS:
        column = component.replace("extent_", "extent_").replace("density_", "density_") + "_numerical"
        column = {"extent_right": "extent_right_numerical",
                  "density_right": "density_right_numerical",
                  "extent_left": "extent_left_numerical",
                  "density_left": "density_left_numerical"}[component]
        values = subset[column].to_numpy()
        valid = ~pd.isna(values)
        components[component] = torch.tensor(
            np.nan_to_num(values, nan=0).astype(np.int64), dtype=torch.long)
        masks[component] = torch.tensor(valid, dtype=torch.bool)
    return covid, covid_mask, components, masks


def decode_predictions(distributions, covid_probability):
    # Hard integer per component (argmax) plus the soft expectation for the continuous score.
    hard, expected, variance = {}, {}, {}
    for component, distribution in distributions.items():
        values = torch.arange(distribution.shape[1], dtype=torch.float32,
                              device=distribution.device)
        hard[component] = distribution.argmax(dim=1)
        mean = (distribution * values).sum(dim=1)
        expected[component] = mean
        variance[component] = (distribution * (values - mean[:, None]) ** 2).sum(dim=1)
    mrale_right = hard["extent_right"] * hard["density_right"]
    mrale_left = hard["extent_left"] * hard["density_left"]
    expected_total = (expected["extent_right"] * expected["density_right"]
                      + expected["extent_left"] * expected["density_left"])
    uncertainty = sum(variance.values()) / len(variance)
    return {
        "hard": hard,
        "mrale_right": mrale_right,
        "mrale_left": mrale_left,
        "mrale_total": mrale_right + mrale_left,
        "mrale_total_expected": expected_total,
        "uncertainty": uncertainty,
        "covid_score": covid_probability,
    }


def positive_weight(subset):
    if not USE_CLASS_BALANCED_LOSS:
        return None
    positives = int((subset["covid_positive"] == "Yes").sum())
    negatives = int((subset["covid_positive"] == "No").sum())
    if positives == 0 or negatives == 0:
        return None
    return torch.tensor(positives / negatives, dtype=torch.float32)

In [ ]:
def run_head_arm(arm, fold, train_subset, validation_subset, test_subset):
    is_lung_arm = arm == "E0f_ordinal_lungs"
    hidden_dim = None if arm == "E0d_linear_v0" else HEAD_HIDDEN_DIM
    dropout = 0.0 if arm == "E0d_linear_v0" else HEAD_DROPOUT
    generator = torch.Generator().manual_seed(SEED + fold)
    torch.manual_seed(SEED + fold)

    def pack(subset):
        keys = subset["image_key"].astype(str).tolist()
        # Ground truth is FIXED for a fold, so extract it once. Calling subset.iloc[i] inside
        # the per-epoch scoring loop meant rows x epochs row materialisations on a wide
        # mixed-dtype frame, and pandas rebuilds an interleaved dtype on every call -- which is
        # exactly where an interrupted run was found sitting.
        records = subset.to_dict("records")
        truth = [ground_truth_fields(record) for record in records]
        covid, covid_mask, components, masks = label_tensors(subset)
        payload = {"keys": keys, "truth": truth, "records": records,
                   "covid": covid, "covid_mask": covid_mask,
                   "components": components, "masks": masks}
        if is_lung_arm:
            payload["left"] = features_for("v2_left", keys)
            payload["right"] = features_for("v2_right", keys)
        else:
            payload["features"] = features_for("v0", keys)
        return payload

    train_data, validation_data, test_data = (
        pack(train_subset), pack(validation_subset), pack(test_subset))
    in_features = (train_data["left"] if is_lung_arm else train_data["features"]).shape[1]

    model = (LungOrdinalHead(in_features, hidden_dim, dropout) if is_lung_arm
             else FrozenHead(in_features, hidden_dim, dropout)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LEARNING_RATE,
                                  weight_decay=HEAD_WEIGHT_DECAY)
    pos_weight = positive_weight(train_subset)
    if pos_weight is not None:
        pos_weight = pos_weight.to(device)

    def forward(data, indices, training):
        if is_lung_arm:
            hidden_left = model.encode(data["left"][indices].to(device))
            hidden_right = model.encode(data["right"][indices].to(device))
            left = model.lung_outputs(hidden_left)
            right = model.lung_outputs(hidden_right)
            logits = {
                "extent_left": left["extent"], "density_left": left["density"],
                "extent_right": right["extent"], "density_right": right["density"],
                "covid": model.covid_logit(hidden_left, hidden_right),
            }
        else:
            logits = model(data["features"][indices].to(device))
        return logits

    def compute_loss(data, indices, logits):
        total = 0.0
        covid_mask = data["covid_mask"][indices].to(device)
        if covid_mask.any():
            total = total + F.binary_cross_entropy_with_logits(
                logits["covid"][covid_mask],
                data["covid"][indices].to(device)[covid_mask],
                pos_weight=pos_weight,
            )
        for component, n_classes in COMPONENTS.items():
            mask = data["masks"][component][indices].to(device)
            if not mask.any():
                continue
            total = total + CoralBlock.loss(
                logits[component][mask],
                data["components"][component][indices].to(device)[mask],
                n_classes,
            )
        return total

    @torch.no_grad()
    def predict(data, mc_dropout=False):
        model.train(mc_dropout)
        n = len(data["keys"])
        samples = MC_DROPOUT_SAMPLES if (mc_dropout and dropout > 0) else 1
        accumulated = None
        covid_accumulated = None
        for _ in range(samples):
            distributions, covid_probability = {}, []
            for offset in range(0, n, 1024):
                indices = torch.arange(offset, min(offset + 1024, n))
                logits = forward(data, indices, training=False)
                for component in COMPONENTS:
                    block = CoralBlock.to_distribution(logits[component])
                    distributions.setdefault(component, []).append(block.cpu())
                covid_probability.append(torch.sigmoid(logits["covid"]).cpu())
            distributions = {key: torch.cat(value) for key, value in distributions.items()}
            covid_probability = torch.cat(covid_probability)
            if accumulated is None:
                accumulated = {key: value.clone() for key, value in distributions.items()}
                covid_accumulated = covid_probability.clone()
            else:
                for key in accumulated:
                    accumulated[key] += distributions[key]
                covid_accumulated += covid_probability
        accumulated = {key: value / samples for key, value in accumulated.items()}
        for key, value in accumulated.items():
            accumulated[key] = value / value.sum(dim=1, keepdim=True).clamp_min(1e-9)
        model.eval()
        return decode_predictions(accumulated, covid_accumulated / samples)

    def selection_score(data):
        decoded = predict(data)
        rows = []
        for index, key in enumerate(data["keys"]):
            rows.append({
                **data["truth"][index],
                "mrale_total": int(decoded["mrale_total"][index]),
                "covid_pred": "Yes" if float(decoded["covid_score"][index]) >= 0.5 else "No",
                "covid_score": float(decoded["covid_score"][index]),
            })
        mrale = cm.mrale_metrics([row for row in rows if row["gt_mrale_total"] is not None])
        covid = cm.classification_metrics(
            [row["gt_covid"] for row in rows], [row["covid_pred"] for row in rows],
            [row["covid_score"] for row in rows])
        mae = mrale.get("mae", float("inf"))
        auroc = covid.get("auroc", float("nan"))
        # Composite: MAE in mRALE points, plus (1-AUROC) scaled to a comparable magnitude
        # (24-point mRALE range / 1.0 AUROC range, damped) so neither task is optimised away.
        penalty = 0.0 if math.isnan(auroc) else (1.0 - auroc) * 8.0
        return mae + penalty, mae, auroc

    n_train = len(train_data["keys"])
    best_score, best_state, best_epoch, stale = float("inf"), None, -1, 0
    history = []
    for epoch in range(HEAD_EPOCHS):
        model.train()
        permutation = torch.randperm(n_train, generator=generator)
        epoch_loss = 0.0
        for offset in range(0, n_train, HEAD_BATCH_SIZE):
            indices = permutation[offset:offset + HEAD_BATCH_SIZE]
            optimizer.zero_grad(set_to_none=True)
            loss = compute_loss(train_data, indices, forward(train_data, indices, True))
            loss.backward()
            optimizer.step()
            epoch_loss += float(loss) * len(indices)
        score, mae, auroc = selection_score(validation_data)
        history.append({"epoch": epoch, "train_loss": epoch_loss / n_train,
                        "val_composite": score, "val_mae": mae, "val_auroc": auroc})
        if score < best_score - 1e-6:
            best_score, best_epoch, stale = score, epoch, 0
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
        else:
            stale += 1
            if stale >= HEAD_PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    decoded = predict(test_data, mc_dropout=True)

    rows = []
    for index, key in enumerate(test_data["keys"]):
        source = test_data["records"][index]
        score = float(decoded["covid_score"][index])
        rows.append(cm.make_prediction_row(
            image_key=key, cohort=source.get("cohort", "MIDRC"),
            subcohort=source.get("subcohort", "MIDRC"), filename=source["filename"],
            held_out_fold=fold, agent="A5_cxformer", arm=arm,
            view="v2_left+v2_right" if is_lung_arm else "v0",
            task="joint",
            covid_pred="Yes" if score >= 0.5 else "No", covid_score=score,
            mrale_total=int(decoded["mrale_total"][index]),
            mrale_right=int(decoded["mrale_right"][index]),
            mrale_left=int(decoded["mrale_left"][index]),
            extent_right=int(decoded["hard"]["extent_right"][index]),
            density_right=int(decoded["hard"]["density_right"][index]),
            extent_left=int(decoded["hard"]["extent_left"][index]),
            density_left=int(decoded["hard"]["density_left"][index]),
            mrale_total_expected=float(decoded["mrale_total_expected"][index]),
            mrale_uncertainty=float(decoded["uncertainty"][index]),
            valid=True, parse_error=None,
            model_id=ENCODER_ID, model_revision=ENCODER_REVISION,
            **test_data["truth"][index],
        ))
    return rows, {"best_epoch": best_epoch, "best_val_composite": best_score,
                  "epochs_run": len(history), "history": history}, model

In [ ]:
PREDICTION_PATH = NB05_DIR / "predictions_frozen.jsonl"
predictions = []
per_fold_metrics = defaultdict(dict)
fold_diagnostics = {}
trained_heads = {}

for arm in RUN_ARMS:
    print("=" * 78)
    print("ARM:", arm)
    arm_rows = []
    for fold in range(N_FOLDS):
        test_subset = internal_cohort[internal_cohort["fold"] == fold]
        pool = internal_cohort[internal_cohort["fold"] != fold]
        train_subset, validation_subset = grouped_inner_split(
            pool, INTERNAL_VALIDATION_FRACTION, SEED + fold)
        started = time.perf_counter()
        rows, diagnostics, model = run_head_arm(
            arm, fold, train_subset, validation_subset, test_subset)
        elapsed = time.perf_counter() - started
        for row in rows:
            row["seconds"] = round(elapsed / max(len(rows), 1), 6)
        arm_rows.extend(rows)
        trained_heads[(arm, fold)] = model
        metrics = evaluate_arm(rows, f"{arm}/fold{fold}")
        per_fold_metrics[arm][fold] = metrics
        fold_diagnostics[f"{arm}/fold{fold}"] = {
            key: value for key, value in diagnostics.items() if key != "history"}
        print(f"  fold {fold}: train={len(train_subset)} val={len(validation_subset)} "
              f"test={len(test_subset)} | best_epoch={diagnostics['best_epoch']} "
              f"| {elapsed:.1f}s")
        print_arm_summary(metrics)
    predictions.extend(arm_rows)
    print()
    print("  POOLED out-of-fold:")
    print_arm_summary(evaluate_arm(arm_rows, f"{arm}/pooled"))

cm.write_jsonl(PREDICTION_PATH, predictions)
print()
print(f"Wrote {len(predictions):,} predictions to {PREDICTION_PATH}")

## 7. Metrics, fold aggregation, and the arm summary

Pooled out-of-fold point estimates are the headline; fold-level means with a
t-distribution 95% CI follow the convention already used by
`cross_validation_aggregate_95ci.csv`, so old and new tables line up column for column.

In [ ]:
summary_rows = []
aggregate_frames = {}

for arm in RUN_ARMS:
    arm_rows = [row for row in predictions if row["arm"] == arm]
    pooled = evaluate_arm(arm_rows, arm)
    aggregate = cm.aggregate_over_folds(
        {fold: metrics for fold, metrics in per_fold_metrics[arm].items()})
    aggregate_frames[arm] = pd.DataFrame(aggregate)

    covid = pooled.get("covid", {})
    mrale = pooled.get("mrale", {})

    def fold_ci(metric):
        match = [row for row in aggregate if row["metric"] == metric]
        if not match:
            return None, None, None
        row = match[0]
        return row["mean"], row["ci95_lower"], row["ci95_upper"]

    auroc_mean, auroc_low, auroc_high = fold_ci("covid.auroc")
    mae_mean, mae_low, mae_high = fold_ci("mrale.mae")
    summary_rows.append(OrderedDict([
        ("arm", arm),
        ("agent", "A5_cxformer"),
        ("n_images", pooled["n_rows"]),
        # PRIMARY ENDPOINTS
        ("mrale_mae_pooled", round(mrale.get("mae", float("nan")), 3)),
        ("mrale_mae_fold_mean", None if mae_mean is None else round(mae_mean, 3)),
        ("mrale_mae_ci95", None if mae_mean is None else f"[{mae_low:.3f}, {mae_high:.3f}]"),
        ("covid_auroc_pooled", round(covid.get("auroc", float("nan")), 4)),
        ("covid_auroc_fold_mean", None if auroc_mean is None else round(auroc_mean, 4)),
        ("covid_auroc_ci95", None if auroc_mean is None else f"[{auroc_low:.4f}, {auroc_high:.4f}]"),
        # SECONDARY
        ("covid_auprc", round(covid.get("auprc", float("nan")), 4)),
        ("covid_balanced_accuracy", round(covid.get("balanced_accuracy", float("nan")), 4)),
        ("covid_sensitivity", round(covid.get("sensitivity", float("nan")), 4)),
        ("covid_specificity", round(covid.get("specificity", float("nan")), 4)),
        ("covid_f1", round(covid.get("f1", float("nan")), 4)),
        ("covid_mcc", round(covid.get("mcc", float("nan")), 4)),
        ("covid_brier", round(covid.get("brier", float("nan")), 4)),
        ("covid_ece", round(covid.get("ece", float("nan")), 4)),
        ("mrale_rmse", round(mrale.get("rmse", float("nan")), 3)),
        ("mrale_qwk", round(mrale.get("qwk", float("nan")), 4)),
        ("mrale_spearman", round(mrale.get("spearman_rho", float("nan")), 4)),
        ("mrale_within1", round(mrale.get("within1_accuracy", float("nan")), 4)),
        ("mrale_exact", round(mrale.get("exact_accuracy", float("nan")), 4)),
        ("mrale_band_accuracy", round(mrale.get("band_accuracy", float("nan")), 4)),
        ("mrale_formula_consistency", round(mrale.get("formula_consistency", float("nan")), 4)),
        ("mae_band_none", round(mrale.get("mae_band_none", float("nan")), 3)),
        ("mae_band_mild", round(mrale.get("mae_band_mild", float("nan")), 3)),
        ("mae_band_moderate", round(mrale.get("mae_band_moderate", float("nan")), 3)),
        ("mae_band_severe", round(mrale.get("mae_band_severe", float("nan")), 3)),
        ("coverage", round(mrale.get("coverage", float("nan")), 4)),
    ]))
    aggregate_frames[arm].to_csv(
        NB05_DIR / f"cross_validation_aggregate_95ci_{arm}.csv", index=False)

summary = pd.DataFrame(summary_rows)
summary.to_csv(NB05_DIR / "arm_summary.csv", index=False)
cm.write_json(NB05_DIR / "per_fold_metrics.json",
              {arm: per_fold_metrics[arm] for arm in RUN_ARMS})

pd.set_option("display.width", 200)
print(summary[[
    "arm", "mrale_mae_pooled", "mrale_mae_ci95", "covid_auroc_pooled", "covid_auroc_ci95",
    "covid_balanced_accuracy", "covid_specificity", "mrale_qwk",
]].to_string(index=False))
print()
print("Severity-stratified mRALE MAE:")
print(summary[["arm", "mae_band_none", "mae_band_mild", "mae_band_moderate",
               "mae_band_severe"]].to_string(index=False))

### Reading E0f against E0d/E0e

If `E0f_ordinal_lungs` beats `E0e_mlp_v0` on mRALE MAE beyond its confidence interval, the
anatomy-aware decomposition is helping a frozen, non-generative model with a **shared** per-lung
head — i.e. it is supplying a real inductive bias, not compensating for a generative model's
difficulty at counting. That is the strongest available support for the novelty claim in
protocol Section 6.2 E4 / referee 2a.

If E0f does *not* beat E0e here but the generative E4d arm does beat E4c in NB 11, the honest
reading is narrower: the decomposition helps *generative* predictors specifically. Say that,
rather than the stronger claim.

## 8. External cohorts

Frozen heads are cheap, so all five fold heads predict every external image and their outputs
are averaged. This is the protocol's `external_ensemble` mode and is legitimate **only**
because no external image was in any training fold — NB 03's membership guard is what licenses
it. The same averaging over internal images would be a leakage result and is not done.

In [ ]:
external_predictions = []

if RUN_EXTERNAL and len(external_cohort):
    print(f"External images: {len(external_cohort):,} "
          f"({dict(Counter(external_cohort['cohort']))})")
    for arm in RUN_ARMS:
        is_lung_arm = arm == "E0f_ordinal_lungs"
        keys = external_cohort["image_key"].astype(str).tolist()
        if is_lung_arm:
            available = [key for key in keys
                         if key in embeddings["v2_left"] and key in embeddings["v2_right"]]
        else:
            available = [key for key in keys if key in embeddings["v0"]]
        if not available:
            print(f"  {arm}: no external embeddings available; skipped")
            continue
        index_by_key = {key: index for index, key in enumerate(keys)}
        external_records = external_cohort.to_dict("records")

        accumulated_total, accumulated_score = None, None
        accumulated_components = None
        for fold in range(N_FOLDS):
            model = trained_heads.get((arm, fold))
            if model is None:
                continue
            model.eval()
            with torch.no_grad():
                if is_lung_arm:
                    left = features_for("v2_left", available).to(device)
                    right = features_for("v2_right", available).to(device)
                    hidden_left, hidden_right = model.encode(left), model.encode(right)
                    left_out, right_out = model.lung_outputs(hidden_left), model.lung_outputs(hidden_right)
                    logits = {
                        "extent_left": left_out["extent"], "density_left": left_out["density"],
                        "extent_right": right_out["extent"], "density_right": right_out["density"],
                        "covid": model.covid_logit(hidden_left, hidden_right),
                    }
                else:
                    logits = model(features_for("v0", available).to(device))
                distributions = {
                    component: CoralBlock.to_distribution(logits[component]).cpu()
                    for component in COMPONENTS
                }
                score = torch.sigmoid(logits["covid"]).cpu()
            if accumulated_components is None:
                accumulated_components = {k: v.clone() for k, v in distributions.items()}
                accumulated_score = score.clone()
            else:
                for key in accumulated_components:
                    accumulated_components[key] += distributions[key]
                accumulated_score += score

        n_models = sum(1 for fold in range(N_FOLDS) if (arm, fold) in trained_heads)
        accumulated_components = {k: v / n_models for k, v in accumulated_components.items()}
        decoded = decode_predictions(accumulated_components, accumulated_score / n_models)

        for position, key in enumerate(available):
            source = external_records[index_by_key[key]]
            score = float(decoded["covid_score"][position])
            external_predictions.append(cm.make_prediction_row(
                image_key=key, cohort=source.get("cohort"),
                subcohort=source.get("subcohort"), filename=source["filename"],
                held_out_fold=None, agent="A5_cxformer", arm=arm,
                view="v2_left+v2_right" if is_lung_arm else "v0", task="joint",
                covid_pred="Yes" if score >= 0.5 else "No", covid_score=score,
                mrale_total=int(decoded["mrale_total"][position]),
                mrale_right=int(decoded["mrale_right"][position]),
                mrale_left=int(decoded["mrale_left"][position]),
                extent_right=int(decoded["hard"]["extent_right"][position]),
                density_right=int(decoded["hard"]["density_right"][position]),
                extent_left=int(decoded["hard"]["extent_left"][position]),
                density_left=int(decoded["hard"]["density_left"][position]),
                mrale_total_expected=float(decoded["mrale_total_expected"][position]),
                mrale_uncertainty=float(decoded["uncertainty"][position]),
                valid=True, parse_error=None,
                model_id=ENCODER_ID, model_revision=ENCODER_REVISION,
                ensemble_of_folds=n_models,
                **ground_truth_fields(source),
            ))
        print(f"  {arm}: {len(available)} external predictions "
              f"(ensemble of {n_models} fold heads)")

    cm.write_jsonl(NB05_DIR / "external_predictions.jsonl", external_predictions)

    external_summary = []
    for arm in RUN_ARMS:
        for cohort_id in sorted({row["cohort"] for row in external_predictions
                                 if row["arm"] == arm}):
            subset = [row for row in external_predictions
                      if row["arm"] == arm and row["cohort"] == cohort_id]
            for subcohort in sorted({row["subcohort"] for row in subset}):
                rows = [row for row in subset if row["subcohort"] == subcohort]
                metrics = evaluate_arm(rows, f"{arm}/{subcohort}")
                covid = metrics.get("covid", {})
                external_summary.append(OrderedDict([
                    ("arm", arm), ("cohort", cohort_id), ("subcohort", subcohort),
                    ("n", len(rows)),
                    ("specificity", round(covid.get("specificity", float("nan")), 4)),
                    ("false_positive_rate",
                     round(1 - covid.get("specificity", float("nan")), 4)),
                    ("mean_predicted_mrale",
                     round(float(np.mean([row["mrale_total"] for row in rows])), 2)),
                ]))
    if external_summary:
        external_frame = pd.DataFrame(external_summary)
        external_frame.to_csv(NB05_DIR / "external_summary.csv", index=False)
        print()
        print(external_frame.to_string(index=False))
        print()
        print("X1a_normal specificity is the headline external number here (protocol E9a).")
        print("X1b_tb shows whether tuberculous opacity is misread as COVID (E9b) -- and the")
        print("mean predicted mRALE on TB cases is informative even without a severity label.")
else:
    print("External prediction skipped (RUN_EXTERNAL False or no external cohort).")

## 9. Run configuration and gate

In [ ]:
run_config = {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "05_frozen_encoder_agents.ipynb",
    "protocol_experiments": ["E0d", "E0e", "E0f"],
    "agent": "A5_cxformer",
    "seed": SEED,
    "encoder": {
        "model_id": ENCODER_ID, "revision": ENCODER_REVISION, "backend": encoder_backend,
        "parameters": int(n_encoder_parameters), "frozen": True,
        "pooling": EMBEDDING_POOLING, "embedding_dim": int(embedding_dim),
    },
    "head": {
        "epochs": HEAD_EPOCHS, "batch_size": HEAD_BATCH_SIZE,
        "learning_rate": HEAD_LEARNING_RATE, "weight_decay": HEAD_WEIGHT_DECAY,
        "hidden_dim": HEAD_HIDDEN_DIM, "dropout": HEAD_DROPOUT, "patience": HEAD_PATIENCE,
        "mc_dropout_samples": MC_DROPOUT_SAMPLES,
        "ordinal_scheme": "CORAL cumulative-link, shared weights with ordered thresholds",
        "selection_criterion": "inner-validation mRALE MAE + 8*(1 - AUROC)",
        "trainable_parameters_linear": int(sum(
            p.numel() for p in FrozenHead(embedding_dim).parameters())),
    },
    "imbalance_hooks": {
        "class_balanced_loss": USE_CLASS_BALANCED_LOSS,
        "logit_adjustment": USE_LOGIT_ADJUSTMENT,
        "note": "Off by default; NB 08 owns the E8 imbalance sweep.",
    },
    "data": {
        "fold_definitions": str(FOLD_DEF_DIR / "midrc_folds_v2.csv"),
        "view_index": str(NB04_DIR / "view_index.csv"),
        "internal_images": int(len(internal_cohort)),
        "external_images": int(len(external_cohort)),
        "inner_validation_fraction": INTERNAL_VALIDATION_FRACTION,
    },
    "fold_diagnostics": fold_diagnostics,
    "smoke_test_images": SMOKE_TEST_IMAGES,
}
cm.write_json(NB05_DIR / "run_config.json", run_config)

failures, warnings = [], []

for view_name in EMBEDDING_VIEWS:
    expected = {str(key) for key in internal_cohort["image_key"]}
    have = set(embeddings[view_name])
    missing = expected - have
    if missing:
        failures.append(f"[{view_name}] missing embeddings for {len(missing)} internal images "
                        f"(e.g. {sorted(missing)[:3]}).")

for arm in RUN_ARMS:
    arm_rows = [row for row in predictions if row["arm"] == arm]
    covered = {row["image_key"] for row in arm_rows}
    expected = {str(key) for key in internal_cohort["image_key"]}
    if covered != expected:
        failures.append(
            f"{arm}: out-of-fold coverage is {len(covered)} of {len(expected)} internal "
            "images. Every image must be predicted exactly once, by the fold that held it out."
        )
    counts = Counter(row["image_key"] for row in arm_rows)
    repeated = [key for key, count in counts.items() if count > 1]
    if repeated:
        failures.append(f"{arm}: {len(repeated)} images predicted more than once "
                        f"(e.g. {repeated[:3]}). That would be a leakage result.")
    for fold, metrics in per_fold_metrics[arm].items():
        auroc = metrics.get("covid", {}).get("auroc")
        if auroc is None or (isinstance(auroc, float) and math.isnan(auroc)):
            failures.append(f"{arm} fold {fold}: AUROC not computable. Endpoint P2 requires "
                            "both PCR classes in every fold; check NB 02's fold balance.")
    agreement = evaluate_arm(arm_rows, arm).get("covid", {}).get("score_decision_agreement")
    if agreement is not None and not math.isnan(agreement) and agreement < 0.995:
        warnings.append(f"{arm}: continuous score reproduces the hard decision on only "
                        f"{agreement:.4f} of cases (protocol 7.2 wants >= 0.995).")

if SMOKE_TEST_IMAGES is not None:
    warnings.append(f"SMOKE TEST MODE: only {SMOKE_TEST_IMAGES} images. Set "
                    "SMOKE_TEST_IMAGES=None and re-run before quoting any number.")

pooled_by_arm = {arm: evaluate_arm([row for row in predictions if row["arm"] == arm], arm)
                 for arm in RUN_ARMS}
best_arm = min(pooled_by_arm, key=lambda arm: pooled_by_arm[arm]["mrale"]["mae"])
best_mae = pooled_by_arm[best_arm]["mrale"]["mae"]
best_auroc = pooled_by_arm[best_arm].get("covid", {}).get("auroc", float("nan"))
warnings.append(
    f"GATE G2 INPUT: best frozen arm is {best_arm} at mRALE MAE {best_mae:.3f}, "
    f"AUROC {best_auroc:.4f}. Compare against the LoRA arms in NB 08/09. If a frozen 87M "
    "probe is competitive with a 4B LoRA VLM, the paper must report that, not bury it."
)


def report(title, messages):
    print(title)
    if messages:
        for message in messages:
            print("  -", message)
    else:
        print("  none")


report("WARNINGS", warnings)
print()
report("FAILURES", failures)

cm.write_json(NB05_DIR / "gate_nb05.json", {
    "passed": not failures, "failures": failures, "warnings": warnings,
    "best_arm": best_arm, "best_mae": best_mae, "best_auroc": best_auroc,
})

assert not failures, f"NB 05 gate failed with {len(failures)} blocking issue(s)."
print()
print("NB 05 gate: PASSED")

## Notes carried forward

- `predictions_frozen.jsonl` and `external_predictions.jsonl` use the shared schema in
  `cxr_metrics.PREDICTION_FIELDS`. NB 12 (agent registry) reads them directly; no
  arm-specific parsing.
- **Agent A5's registry contribution is the best of E0d/E0e/E0f**, selected on inner
  validation, never on the test folds. Record which arm was chosen in the registry so E1-L5's
  leave-one-out is interpretable.
- `mrale_total_expected` and `mrale_uncertainty` are populated here, so this agent can feed
  E3c (per-case uncertainty supplied to the reasoner) without extra work.
- Head trainable-parameter counts are in `run_config.json` for Table 10's operational column.
  The contrast worth printing: ~87M frozen encoder plus a few hundred thousand head
  parameters, against ~4B with a rank-32 LoRA.
- If CXformer fails to load on Biowulf, fix it rather than substituting a backbone. It is the
  only frozen-encoder arm since cxr-foundation was withdrawn (D2a), so RQ3 has no fallback.